In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Fetch the data

In [ ]:
import tomllib
import sys
from datetime import datetime, timezone

user_name = spark.sql("SELECT current_user()").collect()[0][0]
sys.path.insert(0, f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting")

config_path = f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting/config.toml"
with open(config_path, "rb") as f:
    config = tomllib.load(f)

from_dt = datetime.combine(config["dates"]["historical"]["from_date"], datetime.min.time()).replace(tzinfo=timezone.utc)
to_dt   = datetime.combine(config["dates"]["historical"]["to_date"], datetime.min.time()).replace(tzinfo=timezone.utc)

from src.data.collectors.carbon_intensity import fetch_regional_carbon_intensity_by_region_in_chunks


### Fetch Data in Chunks

Fetching ~180 days of data in 7 day chunks

In [ ]:
df_bronze = fetch_regional_carbon_intensity_by_region_in_chunks(from_dt, to_dt)
display(df_bronze.limit(50))


In [0]:
display(df_bronze.limit(50))

In [0]:
print(f"Rows: {df_bronze.count()}")
print(f"Columns: {len(df_bronze.columns)}") 

### Write a Delta Table for the Bronze layer

In [ ]:
sdf = spark.createDataFrame(df_bronze)

sdf.createOrReplaceTempView("carbon_intensity_staging")
spark.sql("CREATE TABLE IF NOT EXISTS bronze_carbon_intensity_regional USING DELTA AS SELECT * FROM carbon_intensity_staging WHERE 1=0")

spark.sql("""
MERGE INTO bronze_carbon_intensity_regional
  USING carbon_intensity_staging
  ON bronze_carbon_intensity_regional.region_id = carbon_intensity_staging.region_id
  AND bronze_carbon_intensity_regional.period_from = carbon_intensity_staging.period_from
  WHEN NOT MATCHED THEN INSERT *
""")
